# Atlas Fulfillment: Example Analysis

This notebook walks through the synthetic `atlas-fulfillment` running example that ships with `oceldb`.

The dataset models a multi-object fulfillment and reverse-logistics process with these object types:

- `customer`
- `order`
- `order_item`
- `package`
- `shipment`
- `invoice`
- `payment`
- `return_case`
- `supplier_order`

The process includes fraud review, backorders, supplier replenishment, split packing, split shipping, customs holds, delayed payment, returns, inspection, restocking, scrapping, and refunds.

The notebook keeps the main analysis path fast and uses only the public `oceldb` API. A final optional section shows heavier relation-based predicates that are useful, but noticeably slower on larger logs.

In [21]:
from pathlib import Path
from pprint import pprint

from oceldb import OCEL, asc, avg, col, count, desc


def find_dataset() -> Path:
    relative_candidates = [
        Path("examples/data/atlas-fulfillment"),
        Path("data/atlas-fulfillment"),
        Path("atlas-fulfillment"),
    ]

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        for relative in relative_candidates:
            candidate = (root / relative).resolve()
            if candidate.exists():
                return candidate

    tried = [str((root / relative).resolve()) for root in search_roots for relative in relative_candidates]
    raise FileNotFoundError("Could not locate atlas-fulfillment. Tried:\n" + "\n".join(tried))


## Load The Running Example

This cell locates the dataset directory, opens it with `OCEL.read(...)`, and keeps the handle open for the rest of the notebook.

In [22]:
SOURCE = find_dataset()

try:
    ocel.close()
except Exception:
    pass

ocel = OCEL.read(SOURCE)
print(f"Loaded: {SOURCE}")
print(ocel)
print(ocel.manifest)

Loaded: /Users/smenne/Developer/Projects/Private/oceldb/examples/data/atlas-fulfillment
OCEL(path='atlas-fulfillment')
OCELManifest(oceldb_version='0.3.0', storage_version='3', source='atlas-fulfillment.sqlite', created_at=datetime.datetime(2026, 4, 12, 21, 28, 34, 458978, tzinfo=datetime.timezone.utc), tables={'event': TableSchema(name='event', core_columns={'ocel_id': 'VARCHAR', 'ocel_type': 'VARCHAR', 'ocel_time': 'TIMESTAMP'}, custom_columns={'amount': 'DOUBLE', 'attempt_no': 'BIGINT', 'carrier': 'VARCHAR', 'channel': 'VARCHAR', 'clearance_channel': 'VARCHAR', 'country': 'VARCHAR', 'cycle_days': 'DOUBLE', 'days_after_delivery': 'BIGINT', 'days_to_pay': 'BIGINT', 'decision': 'VARCHAR', 'delivery_days': 'DOUBLE', 'disposition': 'VARCHAR', 'expected_delay_days': 'BIGINT', 'extra_delay_days': 'BIGINT', 'final_status': 'VARCHAR', 'gross_amount': 'DOUBLE', 'hold_reason': 'VARCHAR', 'hub': 'VARCHAR', 'item_count': 'BIGINT', 'lane': 'VARCHAR', 'lead_time_days': 'BIGINT', 'method': 'VARCHAR

## Structural Overview

Start with the built-in inspection layer. These helpers are implemented on top of the DSL and give a quick feel for scale and object-centric structure.

In [7]:
from oceldb.inspect import overview, event_object_stats

overview(ocel), event_object_stats(ocel)

(OCELOverview(event_count=46585, object_count=23746, event_type_count=27, object_type_count=9, earliest_event_time=datetime.datetime(2024, 1, 1, 8, 0), latest_event_time=datetime.datetime(2024, 6, 24, 5, 37)),
 EventObjectStats(avg_objects_per_event=2.902930127723516, min_objects_per_event=1, max_objects_per_event=7, avg_events_per_object=5.703867729554178, min_events_per_object=1, max_events_per_object=49))

In [10]:
from oceldb.inspect import event_type_counts, object_type_counts

event_type_counts_ = event_type_counts(ocel)
object_type_counts_ = object_type_counts(ocel)

print("Event type counts")
pprint(event_type_counts_)
print()
print("Object type counts")
pprint(object_type_counts_)

Event type counts
{'Approve Return': 714,
 'Backorder Item': 1597,
 'Clear Fraud Review': 561,
 'Close Order': 2500,
 'Create Invoice': 2457,
 'Create Shipment': 2685,
 'Create Supplier Order': 1597,
 'Customs Hold': 279,
 'Customs Release': 279,
 'Deliver Shipment': 2685,
 'Delivery Attempt Failed': 299,
 'Dispatch Shipment': 2685,
 'Inspect Return': 714,
 'Issue Refund': 714,
 'Pack Package': 3596,
 'Pick Item': 6672,
 'Place Order': 2500,
 'Receive Payment': 2576,
 'Receive Return': 714,
 'Receive Supplier Delivery': 1597,
 'Reject Order': 43,
 'Request Return': 714,
 'Reserve Inventory': 6672,
 'Restock Item': 513,
 'Scrap Item': 201,
 'Send Payment Reminder': 417,
 'Start Fraud Review': 604}

Object type counts
{'customer': 833,
 'invoice': 2457,
 'order': 2500,
 'order_item': 6788,
 'package': 3596,
 'payment': 2576,
 'return_case': 714,
 'shipment': 2685,
 'supplier_order': 1597}


In [11]:
from oceldb.inspect import attributes

attribute_map = attributes(ocel)
selected_attributes = {
    "object": {
        "order": attribute_map["object"]["order"],
        "shipment": attribute_map["object"]["shipment"],
        "return_case": attribute_map["object"]["return_case"],
    },
    "event": {
        "Place Order": attribute_map["event"]["Place Order"],
        "Deliver Shipment": attribute_map["event"]["Deliver Shipment"],
        "Issue Refund": attribute_map["event"]["Issue Refund"],
    },
}

pprint(selected_attributes)

{'event': {'Deliver Shipment': ['delivery_days', 'on_time'],
           'Issue Refund': ['refund_amount', 'refund_type'],
           'Place Order': ['channel',
                           'country',
                           'item_count',
                           'order_value',
                           'priority']},
 'object': {'order': ['channel',
                      'country',
                      'order_value',
                      'payment_method',
                      'payment_terms_days',
                      'priority',
                      'risk_bucket',
                      'status'],
            'return_case': ['reason', 'refundable_amount', 'resolution'],
            'shipment': ['attempt_count',
                         'carrier',
                         'final_status',
                         'had_customs_hold',
                         'is_international',
                         'lane']}}


## Event Activity

The first analytical pass is usually event-centric: which event types dominate the log and how activity evolves over time.

In [16]:
top_event_types = (
    ocel.query
    .events()
    .group_by("ocel_type")
    .agg(count().alias("events"))
    .sort(desc("events"), asc("ocel_type"))
    .collect()
)

top_event_types.limit(15)

┌───────────────────────────┬────────┐
│         ocel_type         │ events │
│          varchar          │ int64  │
├───────────────────────────┼────────┤
│ Pick Item                 │   6672 │
│ Reserve Inventory         │   6672 │
│ Pack Package              │   3596 │
│ Create Shipment           │   2685 │
│ Deliver Shipment          │   2685 │
│ Dispatch Shipment         │   2685 │
│ Receive Payment           │   2576 │
│ Close Order               │   2500 │
│ Place Order               │   2500 │
│ Create Invoice            │   2457 │
│ Backorder Item            │   1597 │
│ Create Supplier Order     │   1597 │
│ Receive Supplier Delivery │   1597 │
│ Approve Return            │    714 │
│ Inspect Return            │    714 │
└───────────────────────────┴────────┘
  15 rows                  2 columns

In [20]:
delivery_quality = (
    ocel.query
    .events("Deliver Shipment")
    .group_by("on_time")
    .agg(
        count().alias("shipments"),
        avg(col("delivery_days")).alias("avg_delivery_days"),
    )
    .sort(desc("shipments"))
    .collect()
)

delivery_quality

┌─────────┬───────────┬────────────────────┐
│ on_time │ shipments │ avg_delivery_days  │
│ boolean │   int64   │       double       │
├─────────┼───────────┼────────────────────┤
│ false   │      1914 │  6.926055381400241 │
│ true    │       771 │ 3.4232295719844297 │
└─────────┴───────────┴────────────────────┘

In [23]:
monthly_event_profile = ocel.sql(
    f'''
    SELECT date_trunc('month', ocel_time) AS month,
           ocel_type,
           COUNT(*) AS events
    FROM "event"
    GROUP BY 1, 2
    ORDER BY month, events DESC, ocel_type
    LIMIT 18
    '''
)

monthly_event_profile

┌─────────────────────┬───────────────────────────┬────────┐
│        month        │         ocel_type         │ events │
│      timestamp      │          varchar          │ int64  │
├─────────────────────┼───────────────────────────┼────────┤
│ 2024-01-01 00:00:00 │ Reserve Inventory         │   3606 │
│ 2024-01-01 00:00:00 │ Pick Item                 │   3140 │
│ 2024-01-01 00:00:00 │ Pack Package              │   1708 │
│ 2024-01-01 00:00:00 │ Place Order               │   1360 │
│ 2024-01-01 00:00:00 │ Create Shipment           │   1290 │
│ 2024-01-01 00:00:00 │ Dispatch Shipment         │   1273 │
│ 2024-01-01 00:00:00 │ Deliver Shipment          │    989 │
│ 2024-01-01 00:00:00 │ Create Invoice            │    960 │
│ 2024-01-01 00:00:00 │ Backorder Item            │    845 │
│ 2024-01-01 00:00:00 │ Create Supplier Order     │    845 │
│ 2024-01-01 00:00:00 │ Receive Payment           │    681 │
│ 2024-01-01 00:00:00 │ Receive Supplier Delivery │    663 │
│ 2024-01-01 00:00:00 │ 

## Orders And Commercial Behavior

Now switch to object-rooted analysis. Orders are the natural business anchor for commercial questions such as channel mix, payment behavior, and unusually large cases.

In [25]:
order_status_mix = (
    ocel.query
    .object_states("order")
    .latest()
    .group_by("country", "channel", "status")
    .agg(
        count().alias("orders"),
        avg(col("order_value")).alias("avg_order_value"),
    )
    .sort(desc("orders"), desc("avg_order_value"))
    .collect().limit(15)
)

order_status_mix

┌──────────────────────┬─────────────┬───────────┬────────┬────────────────────┐
│       country        │   channel   │  status   │ orders │  avg_order_value   │
│       varchar        │   varchar   │  varchar  │ int64  │       double       │
├──────────────────────┼─────────────┼───────────┼────────┼────────────────────┤
│ Germany              │ web         │ delivered │    204 │ 1453.2347549019607 │
│ United States        │ web         │ delivered │    136 │ 1408.0468382352938 │
│ Netherlands          │ web         │ delivered │    108 │ 1446.8919444444446 │
│ Belgium              │ web         │ delivered │     98 │ 1393.8163265306123 │
│ Germany              │ marketplace │ delivered │     90 │ 1446.3977777777777 │
│ United Kingdom       │ web         │ delivered │     83 │  1249.011204819277 │
│ Germany              │ mobile      │ delivered │     73 │ 1471.6701369863015 │
│ Sweden               │ web         │ delivered │     73 │ 1362.2598630136986 │
│ Germany              │ sal

In [26]:
payment_mix = (
    ocel.query
    .object_states("payment")
    .latest()
    .group_by("method", "status")
    .agg(
        count().alias("payments"),
        avg(col("amount")).alias("avg_amount"),
    )
    .sort(desc("payments"), desc("avg_amount"))
    .collect()
)

payment_mix

┌─────────┬─────────┬──────────┬────────────────────┐
│ method  │ status  │ payments │     avg_amount     │
│ varchar │ varchar │  int64   │       double       │
├─────────┼─────────┼──────────┼────────────────────┤
│ invoice │ settled │      982 │ 1374.6069551934825 │
│ card    │ settled │      970 │ 1466.7332268041234 │
│ wire    │ settled │      381 │ 1338.7993700787404 │
│ wallet  │ settled │      243 │ 1654.6399176954733 │
└─────────┴─────────┴──────────┴────────────────────┘

In [27]:
high_value_orders = (
    ocel.query
    .object_states("order")
    .latest()
    .select(
        "ocel_id",
        "country",
        "channel",
        "priority",
        "risk_bucket",
        "payment_method",
        "order_value",
        "status",
    )
    .sort(desc("order_value"))
    .limit(15)
    .collect()
)

high_value_orders

┌──────────────┬──────────────────────┬─────────────┬──────────┬─────────────┬────────────────┬─────────────┬──────────────────┐
│   ocel_id    │       country        │   channel   │ priority │ risk_bucket │ payment_method │ order_value │      status      │
│   varchar    │       varchar        │   varchar   │ varchar  │   varchar   │    varchar     │   double    │     varchar      │
├──────────────┼──────────────────────┼─────────────┼──────────┼─────────────┼────────────────┼─────────────┼──────────────────┤
│ order-002251 │ Canada               │ marketplace │ standard │ medium      │ invoice        │     8623.82 │ returned_partial │
│ order-000411 │ Canada               │ web         │ standard │ medium      │ card           │     7510.54 │ delivered        │
│ order-002290 │ United Arab Emirates │ web         │ expedite │ low         │ invoice        │     7466.61 │ delivered        │
│ order-001166 │ Sweden               │ sales_rep   │ standard │ low         │ invoice        │  

## Operations And Exceptions

The same dataset also supports operational analysis around replenishment, shipping complexity, and returns.

In [28]:
supplier_pressure = (
    ocel.query
    .object_states("supplier_order")
    .latest()
    .group_by("supplier", "status")
    .agg(
        count().alias("supplier_orders"),
        avg(col("lead_time_days")).alias("avg_lead_time_days"),
    )
    .sort(desc("supplier_orders"), desc("avg_lead_time_days"))
    .collect()
)

supplier_pressure

┌───────────────┬──────────┬─────────────────┬────────────────────┐
│   supplier    │  status  │ supplier_orders │ avg_lead_time_days │
│    varchar    │ varchar  │      int64      │       double       │
├───────────────┼──────────┼─────────────────┼────────────────────┤
│ Global Source │ received │             442 │ 6.9366515837104075 │
│ Alpha Parts   │ received │             404 │ 6.8861386138613865 │
│ Mekatron      │ received │             385 │  7.103896103896104 │
│ Nordic Supply │ received │             366 │    7.1775956284153 │
└───────────────┴──────────┴─────────────────┴────────────────────┘

In [29]:
shipment_mix = (
    ocel.query
    .object_states("shipment")
    .latest()
    .group_by("carrier", "is_international", "had_customs_hold")
    .agg(
        count().alias("shipments"),
        avg(col("attempt_count")).alias("avg_attempt_count"),
    )
    .sort(desc("shipments"), desc("avg_attempt_count"))
    .collect()
)

shipment_mix

┌────────────┬──────────────────┬──────────────────┬───────────┬────────────────────┐
│  carrier   │ is_international │ had_customs_hold │ shipments │ avg_attempt_count  │
│  varchar   │     boolean      │     boolean      │   int64   │       double       │
├────────────┼──────────────────┼──────────────────┼───────────┼────────────────────┤
│ DHL        │ false            │ false            │       586 │ 1.0938566552901023 │
│ Maersk Air │ true             │ false            │       382 │ 1.1047120418848169 │
│ UPS        │ false            │ false            │       290 │ 1.1137931034482758 │
│ DPD        │ false            │ false            │       261 │ 1.0881226053639848 │
│ FedEx      │ false            │ false            │       249 │  1.144578313253012 │
│ FedEx      │ true             │ false            │       199 │ 1.1256281407035176 │
│ UPS        │ true             │ false            │       180 │ 1.1222222222222222 │
│ DHL        │ true             │ false            │  

In [30]:
return_mix = (
    ocel.query
    .object_states("return_case")
    .latest()
    .group_by("reason", "resolution")
    .agg(
        count().alias("cases"),
        avg(col("refundable_amount")).alias("avg_refundable_amount"),
    )
    .sort(desc("cases"), desc("avg_refundable_amount"))
    .collect()
)

return_mix

┌───────────────┬────────────────┬───────┬───────────────────────┐
│    reason     │   resolution   │ cases │ avg_refundable_amount │
│    varchar    │    varchar     │ int64 │        double         │
├───────────────┼────────────────┼───────┼───────────────────────┤
│ wrong_item    │ refund         │   112 │     322.6691071428572 │
│ size_issue    │ refund         │   105 │     422.0375238095238 │
│ damaged       │ refund         │   105 │    358.62542857142864 │
│ late_delivery │ refund         │   102 │      375.336568627451 │
│ quality_issue │ refund         │    93 │     281.6311827956989 │
│ wrong_item    │ partial_refund │    30 │    293.17600000000004 │
│ late_delivery │ partial_refund │    29 │    428.02827586206894 │
│ damaged       │ partial_refund │    27 │    474.28666666666675 │
│ quality_issue │ partial_refund │    27 │     381.2277777777777 │
│ size_issue    │ partial_refund │    24 │    442.79249999999996 │
│ late_delivery │ store_credit   │    16 │    453.663749999999

## Relation Graph View With Raw SQL

The DSL is the preferred interface for most analysis, but `ocel.sql(...)` is useful for advanced structural summaries that are easier to express as joins. The query below shows which object-type pairs dominate the link graph.

In [31]:
link_graph = ocel.sql(
    f'''
    SELECT source.ocel_type AS source_type,
           target.ocel_type AS target_type,
           COUNT(*) AS links
    FROM "object_object" oo
    JOIN "object" source
      ON oo.ocel_source_id = source.ocel_id
    JOIN "object" target
      ON oo.ocel_target_id = target.ocel_id
    GROUP BY 1, 2
    ORDER BY links DESC, source_type, target_type
    LIMIT 15
    '''
)

link_graph

┌─────────────┬────────────────┬───────┐
│ source_type │  target_type   │ links │
│   varchar   │    varchar     │ int64 │
├─────────────┼────────────────┼───────┤
│ order       │ order_item     │  6788 │
│ order_item  │ package        │  6672 │
│ order       │ package        │  3596 │
│ package     │ shipment       │  3596 │
│ order       │ shipment       │  2685 │
│ invoice     │ payment        │  2576 │
│ order       │ payment        │  2576 │
│ customer    │ order          │  2500 │
│ invoice     │ order          │  2457 │
│ order       │ supplier_order │  1597 │
│ order_item  │ supplier_order │  1597 │
│ order       │ return_case    │   714 │
│ order_item  │ return_case    │   714 │
│ payment     │ return_case    │   714 │
└─────────────┴────────────────┴───────┘
  14 rows                    3 columns

## Optional: Object-Centric Slices With Relation Builders

This final section demonstrates the more OCEL-specific part of the DSL with free relation builders such as `linked(...)` and `has_event(...)`.

These predicates are correlated and can be noticeably slower than the aggregate sections above, especially if you run several of them in one go. Keep them for targeted questions rather than the default analysis path.

In [32]:
from time import perf_counter

from oceldb import has_event, linked


def timed(label: str, fn):
    start = perf_counter()
    value = fn()
    elapsed = perf_counter() - start
    print(f"{label}: {elapsed:.2f}s -> {value}")
    return value


orders = ocel.query.objects("order")


slow_relation_summary = {
    "orders_linked_to_supplier_orders": timed(
        "orders linked to supplier orders",
        lambda: orders.where(linked("supplier_order").exists()).count(),
    ),
    "orders_linked_to_returns": timed(
        "orders linked to return cases",
        lambda: orders.where(linked("return_case").exists()).count(),
    ),
    "orders_with_customs_hold_event": timed(
        "orders with customs hold",
        lambda: orders.where(has_event("Customs Hold").exists()).count(),
    ),
}

slow_relation_summary

orders linked to supplier orders: 0.28s -> 1224
orders linked to return cases: 0.00s -> 624
orders with customs hold: 0.00s -> 270


{'orders_linked_to_supplier_orders': 1224,
 'orders_linked_to_returns': 624,
 'orders_with_customs_hold_event': 270}

## Close The Handle

Close the `OCEL` once you are done. Re-running the load cell will open a fresh handle.

In [33]:
ocel.close()